<a href="https://colab.research.google.com/github/borgesf/nicePythonPlots/blob/main/2026_01_scientificPlots_Python_grayscale_cvd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Creating Scientific Plots in Python — Part 2

This reduced notebook focuses on grayscale checks and colour vision deficiency (CVD) variations.


# Setup

**How to use this cell:**

1. **First time running:**
   When running this notebook for the first time in Colab, remove the `#` from the line

   ```python
   !pip install numpy matplotlib colorspacious
   ```

   to install all required dependencies. You might be asked to restart your session when doing that.


In [ ]:
# @title Installing libraries (click to expand)

# !pip install numpy matplotlib colorspacious


In [ ]:
# @title Importing Libraries (click to expand)

%matplotlib inline

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


In [ ]:
# @title Helper Functions (click to expand)
# @markdown This cell centralizes the helper utilities used in this reduced notebook.
# @markdown
# @markdown **Functions included (in order of appearance):**
# @markdown - `build_grayscale_cmap` — Convert any colormap to a perceptual grayscale (CAM02-UCS J′).
# @markdown - `simulate_partial_color_blind` — Partial red-green deficiency (deuteranomaly; Machado 2009).
# @markdown - `simulate_full_color_blind` — Full green-cone loss (deuteranopia; Brettel 1997).

def build_grayscale_cmap(cmap_in, n=256, name_suffix="_gray"):
    """
    Convert any colormap (name or object) into a grayscale colormap by
    sampling in sRGB, transforming to CAM02-UCS, extracting J′, and
    mapping J′ → (J′, J′, J′).
    """
    # Resolve input to a colormap object
    cmap_obj = mpl.colormaps[cmap_in] if isinstance(cmap_in, str) else cmap_in

    # Local import to avoid hard dependency at cell import time
    from colorspacious import cspace_converter

    # Sample uniformly in sRGB [0..1]
    x = np.linspace(0.0, 1.0, n)
    rgb = cmap_obj(x)[:, :3]  # (n, 3) in sRGB

    # Convert to CAM02-UCS and take J′ (index 0)
    to_cam02ucs = cspace_converter("sRGB1", "CAM02-UCS")
    cam = to_cam02ucs(rgb[np.newaxis, :, :])  # (1, n, 3)
    Jp = cam[0, :, 0]

    # Normalize J′ to [0,1] and build gray ramp
    Jp_norm = (Jp - Jp.min()) / (Jp.max() - Jp.min() + 1e-12)
    gray_triplets = [(v, v, v) for v in Jp_norm]

    # Name for the new grayscale colormap
    base_name = cmap_in if isinstance(cmap_in, str) else getattr(cmap_obj, "name", "cmap")
    gray_name = f"{base_name}{name_suffix}"

    return LinearSegmentedColormap.from_list(gray_name, gray_triplets, N=n)

# -----------------------------
# Partial red-green deficiency (Machado 2009)
# -----------------------------
M_deuteranopia = np.array([
    [0.367322, 0.860646, -0.227968],
    [0.280085, 0.672501,  0.047413],
    [-0.011820, 0.042940,  0.968881]
])

def simulate_partial_color_blind(rgb_array, severity=1.0):
    """
    Simulate partial red-green color deficiency (deuteranomaly).
    Based on Machado et al. 2009.
    """
    # sRGB → linear
    rgb_lin = np.where(
        rgb_array <= 0.04045,
        rgb_array / 12.92,
        ((rgb_array + 0.055) / 1.055) ** 2.4
    )

    # Interpolate transformation by severity
    T = severity * M_deuteranopia + (1 - severity) * np.eye(3)

    # Apply and guard against negatives before gamma
    transformed = np.dot(rgb_lin, T.T)
    transformed = np.clip(transformed, 0.0, None)

    # linear → sRGB
    rgb_sim = np.where(
        transformed <= 0.0031308,
        12.92 * transformed,
        1.055 * (transformed ** (1 / 2.4)) - 0.055
    )

    return np.clip(rgb_sim, 0, 1)

# -----------------------------
# Full green-cone loss (Brettel 1997)
# -----------------------------
def simulate_full_color_blind(rgb_array):
    """
    Simulate full green-cone loss (deuteranopia).
    Based on Brettel et al. 1997.
    """
    def srgb_to_linear(rgb):
        return np.where(
            rgb <= 0.04045,
            rgb / 12.92,
            ((rgb + 0.055) / 1.055) ** 2.4
        )

    linear_rgb = srgb_to_linear(rgb_array)

    rgb_to_lms = np.array([
        [17.8824,   43.5161,   4.11935],
        [ 3.45565,  27.1554,   3.86714],
        [ 0.0299566, 0.184309, 1.46709]
    ])

    lms = np.dot(linear_rgb, rgb_to_lms.T)

    plane1 = np.array([
        [1.0,       0.0,     0.0],
        [0.494207,  0.0,     1.24827],
        [0.0,       0.0,     1.0]
    ])

    plane2 = np.array([
        [1.0,       0.0,     0.0],
        [0.466174,  0.0,     1.44742],
        [0.0,       0.0,     1.0]
    ])

    lms_sim = np.zeros_like(lms)
    mask = lms[:, 0] > lms[:, 1] + lms[:, 2]

    lms_sim[mask]  = np.dot(lms[mask],  plane1.T)
    lms_sim[~mask] = np.dot(lms[~mask], plane2.T)

    lms_to_rgb = np.linalg.inv(rgb_to_lms)
    linear_rgb_sim = np.dot(lms_sim, lms_to_rgb.T)

    def linear_to_srgb(rgb):
        return np.where(
            rgb <= 0.0031308,
            12.92 * rgb,
            1.055 * (rgb ** (1 / 2.4)) - 0.055
        )

    return np.clip(linear_to_srgb(linear_rgb_sim), 0, 1)


# Step 8 — Colour-blind- and grayscale-friendly maps

Goal:
- Ensure that visualizations remain clear and interpretable in grayscale and for viewers with CVD (colour vision deficiencies).


In [ ]:
# -----------------------------
# Synthetic dataset (minimal replacement for the map data)
# -----------------------------
x = np.linspace(0, 10, 200)
y = np.linspace(0, 8, 160)
LON, LAT = np.meshgrid(x, y)

temperature_array = (
    4 * np.sin(LON / 2) +
    3 * np.cos(LAT / 1.5) +
    2 * np.exp(-((LON - 6) ** 2 + (LAT - 4) ** 2) / 6)
)

# -----------------------------
# Colourmap → Grayscale (CAM02-UCS J′-mapped) — using helper
# -----------------------------
# Try cmap = "jet", "inferno", or "viridis"

cmap_in = "jet"  # original colormap to evaluate
cmap_obj = mpl.colormaps[cmap_in] if isinstance(cmap_in, str) else cmap_in
cmap_gray = build_grayscale_cmap(cmap_in, n=256)  # uses helper defined earlier

# -----------------------------
# Side-by-side: original vs grayscale
# -----------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

contour1 = ax1.contourf(LON, LAT, temperature_array, levels=200, cmap=cmap_obj)
ax1.set_title(f"Original ({cmap_in})")
plt.colorbar(contour1, ax=ax1, shrink=0.85)

contour2 = ax2.contourf(LON, LAT, temperature_array, levels=200, cmap=cmap_gray)
ax2.set_title('Grayscale (perceptual)')
plt.colorbar(contour2, ax=ax2, shrink=0.85)

for ax in (ax1, ax2):
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

fig.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Colour Vision Deficiency — compare severity levels
# -----------------------------
cmap_in = "jet"
cmap_obj = mpl.colormaps[cmap_in] if isinstance(cmap_in, str) else cmap_in

# Sample the base colormap and build derived colormaps
x = np.linspace(0.0, 1.0, 256)
rgb = cmap_obj(x)[:, :3]

severities = [0.25, 0.5, 0.75]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, severity in zip(axes, severities):
    rgb_partial = simulate_partial_color_blind(rgb, severity=severity)
    cmap_partial = LinearSegmentedColormap.from_list(
        f"partial_{severity:.2f}", rgb_partial, N=256
    )
    contour = ax.contourf(LON, LAT, temperature_array, levels=200, cmap=cmap_partial)
    ax.set_title(f"Partial CVD (severity={severity:.2f})")
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    plt.colorbar(contour, ax=ax, shrink=0.85)

fig.tight_layout()
plt.show()

# -----------------------------
# Normal vs Partial vs Full
# -----------------------------
rgb_partial = simulate_partial_color_blind(rgb, severity=0.75)
cmap_partial = LinearSegmentedColormap.from_list(
    'partial_0.75', rgb_partial, N=256
)

rgb_full = simulate_full_color_blind(rgb)
cmap_full = LinearSegmentedColormap.from_list(
    'full_deuteranopia', rgb_full, N=256
)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

contour1 = ax1.contourf(LON, LAT, temperature_array, levels=200, cmap=cmap_obj)
ax1.set_title('Normal Vision')
plt.colorbar(contour1, ax=ax1, shrink=0.85)

contour2 = ax2.contourf(LON, LAT, temperature_array, levels=200, cmap=cmap_partial)
ax2.set_title('Partial CVD (0.75)')
plt.colorbar(contour2, ax=ax2, shrink=0.85)

contour3 = ax3.contourf(LON, LAT, temperature_array, levels=200, cmap=cmap_full)
ax3.set_title('Full CVD')
plt.colorbar(contour3, ax=ax3, shrink=0.85)

for ax in (ax1, ax2, ax3):
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

fig.tight_layout()
plt.show()
